# 01 — Análisis Exploratorio (EDA) · CERT r4.2

**Objetivos de este notebook (Fase 1 del PLAN.md):**
1. Validar que el dataset está completo y es legible.
2. Estadísticas por fuente: volumen, rango temporal, nº usuarios.
3. Distribución horaria de la actividad → justificar la definición de "fuera de horario".
4. Cargar el ground truth (70 insiders) y entender los 3 escenarios.
5. Comparar la actividad de un insider contra un usuario normal.

Ejecutable en **Google Colab** (datos en `MyDrive/CERT_data/`) y en **local**.

In [ ]:
# === Setup: entorno, dependencias y rutas ===
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q polars pyarrow
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/CERT_data")
    DATA_DIR = BASE / "r4.2"
    ANSWERS_DIR = BASE / "answers"
    PROCESSED_DIR = BASE / "processed"
else:
    # Local: el notebook vive en SIEM/notebooks/ — reutilizamos src/config.py
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str(PROJECT_ROOT))
    from src.config import DATA_DIR, ANSWERS_DIR, PROCESSED_DIR

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

import polars as pl
import matplotlib.pyplot as plt

DATE_FORMAT = "%m/%d/%Y %H:%M:%S"
WORK_HOURS = (8, 18)

print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")
print(f"Datos:   {DATA_DIR}")
print(f"Answers: {ANSWERS_DIR}")

In [ ]:
# === 1. Validación: ¿están todos los ficheros? ===
expected = ["logon.csv", "device.csv", "email.csv", "file.csv", "psychometric.csv"]

for name in expected:
    f = DATA_DIR / name
    status = f"✅ {f.stat().st_size / 1e6:,.0f} MB" if f.exists() else "❌ FALTA"
    print(f"{name:20s} {status}")

ldap_files = sorted((DATA_DIR / "LDAP").glob("*.csv")) if (DATA_DIR / "LDAP").exists() else []
print(f"{'LDAP/':20s} {'✅ ' + str(len(ldap_files)) + ' snapshots mensuales' if ldap_files else '❌ FALTA'}")

insiders_file = ANSWERS_DIR / "insiders.csv"
print(f"{'answers/insiders.csv':20s} {'✅' if insiders_file.exists() else '❌ FALTA'}")

# http.csv es opcional (fase 6) — solo informativo
http = DATA_DIR / "http.csv"
print(f"{'http.csv (opcional)':20s} {'presente, ' + format(http.stat().st_size/1e9, '.1f') + ' GB' if http.exists() else 'no presente (OK, fase 6)'}")

## 2. Ground truth: los 70 insiders de r4.2

`insiders.csv` etiqueta cada incidente con usuario, escenario y ventana temporal.
Los 3 escenarios de r4.2:

1. **Fuga a wikileaks** — actividad nocturna nueva + USB + subida de datos. Dimite después.
2. **Robo para competidor** — webs de empleo + robo masivo por USB antes de irse.
3. **Sysadmin keylogger** — instala keylogger en el PC del supervisor, suplanta su identidad.

In [ ]:
# OJO: 'dataset' debe leerse como string — si no, Polars lo infiere como float
# (valores 2, 3.1, 4.2...) y el filtro == "4.2" falla con ComputeError.
insiders = (
    pl.read_csv(insiders_file, schema_overrides={"dataset": pl.Utf8})
    .filter(pl.col("dataset") == "4.2")
    .with_columns(
        pl.col("start").str.strptime(pl.Datetime, "%m/%d/%Y %H:%M:%S", strict=False),
        pl.col("end").str.strptime(pl.Datetime, "%m/%d/%Y %H:%M:%S", strict=False),
    )
    .with_columns((pl.col("end") - pl.col("start")).dt.total_days().alias("duracion_dias"))
)

print(f"Total insiders r4.2: {insiders.height}")
print(insiders.group_by("scenario").agg(
    pl.len().alias("n_insiders"),
    pl.col("duracion_dias").mean().round(1).alias("duracion_media_dias"),
).sort("scenario"))

INSIDER_USERS = set(insiders["user"].to_list())
insiders.head(10)

## 3. Logon: volumen, usuarios y distribución horaria

In [ ]:
logon = (
    pl.scan_csv(DATA_DIR / "logon.csv")
    .with_columns(pl.col("date").str.strptime(pl.Datetime, DATE_FORMAT))
    .collect()
)

print(f"Eventos:      {logon.height:,}")
print(f"Usuarios:     {logon['user'].n_unique():,}")
print(f"PCs:          {logon['pc'].n_unique():,}")
print(f"Rango fechas: {logon['date'].min()} → {logon['date'].max()}")
print(logon["activity"].value_counts())

In [ ]:
# Distribución por hora del día → justificación de WORK_HOURS
por_hora = (
    logon.filter(pl.col("activity") == "Logon")
    .group_by(pl.col("date").dt.hour().alias("hora"))
    .agg(pl.len().alias("logins"))
    .sort("hora")
)

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(por_hora["hora"], por_hora["logins"], color="steelblue")
ax.axvspan(WORK_HOURS[0] - 0.5, WORK_HOURS[1] - 0.5, alpha=0.12, color="green",
           label=f"Horario laboral {WORK_HOURS[0]}–{WORK_HOURS[1]}h")
ax.set_xlabel("Hora del día"); ax.set_ylabel("Nº de logins")
ax.set_title("Distribución horaria de logins — toda la organización")
ax.legend(); ax.set_xticks(range(24))
plt.tight_layout(); plt.show()

## 4. Device (USB) y File: las fuentes clave de exfiltración

In [ ]:
device = (
    pl.scan_csv(DATA_DIR / "device.csv")
    .with_columns(pl.col("date").str.strptime(pl.Datetime, DATE_FORMAT))
    .collect()
)
print(f"Eventos USB:           {device.height:,}")
print(f"Usuarios que usan USB: {device['user'].n_unique():,} de {logon['user'].n_unique():,}")

# file.csv: solo columnas ligeras (el content pesa)
file_df = (
    pl.scan_csv(DATA_DIR / "file.csv")
    .select("date", "user", "pc", "filename")
    .with_columns(pl.col("date").str.strptime(pl.Datetime, DATE_FORMAT))
    .collect()
)
print(f"Copias de fichero a USB: {file_df.height:,}")
print(file_df.with_columns(
    pl.col("filename").str.split(".").list.last().alias("extension")
)["extension"].value_counts().sort("count", descending=True).head(8))

## 5. Email: dominios externos (señal de exfiltración)

`email.csv` pesa 1.36 GB — usamos *lazy scan* y **sin** la columna `content`.

In [ ]:
email_stats = (
    pl.scan_csv(DATA_DIR / "email.csv")
    .select("date", "user", "to", "from", "size", "attachments")
    .with_columns(
        pl.col("date").str.strptime(pl.Datetime, DATE_FORMAT),
        # destinatario externo = algún 'to' que no sea @dtaa.com
        pl.col("to").str.contains("@dtaa.com").not_().alias("algun_externo_simple"),
        (~pl.col("to").str.split(";").list.eval(
            pl.element().str.ends_with("@dtaa.com")
        ).list.all()).alias("tiene_destinatario_externo"),
    )
    .select(
        pl.len().alias("total_emails"),
        pl.col("user").n_unique().alias("usuarios"),
        pl.col("tiene_destinatario_externo").mean().alias("pct_con_externo"),
        pl.col("attachments").mean().alias("adjuntos_medios"),
        pl.col("size").mean().alias("tamano_medio_bytes"),
    )
    .collect(engine="streaming")
)
email_stats

## 6. LDAP: organización y bajas de empleados

Snapshots mensuales: un usuario que está en el mes N y no en el N+1 ha causado **baja**.
Los escenarios 1 y 2 terminan con el insider abandonando la organización.

In [ ]:
ldap_files = sorted((DATA_DIR / "LDAP").glob("*.csv"))
primero = pl.read_csv(ldap_files[0])
ultimo = pl.read_csv(ldap_files[-1])

print(f"Snapshot {ldap_files[0].stem}: {primero.height} empleados")
print(f"Snapshot {ldap_files[-1].stem}: {ultimo.height} empleados")
print(f"Columnas: {primero.columns}\n")

# Bajas: presentes al inicio, ausentes al final
bajas = set(primero["user_id"]) - set(ultimo["user_id"])
print(f"Bajas durante el periodo: {len(bajas)}")
print(f"Insiders entre las bajas: {len(bajas & INSIDER_USERS)} de {len(INSIDER_USERS)} insiders")

print("\nRoles más comunes:")
print(primero["role"].value_counts().sort("count", descending=True).head(8))

## 7. Zoom: un insider del escenario 1 vs. su comportamiento previo

El escenario 1 promete: *"usuario sin historial de USB ni actividad nocturna empieza
a hacer ambas cosas"*. Verifiquémoslo visualmente con el primer insider del escenario 1.

In [ ]:
caso = insiders.filter(pl.col("scenario") == 1).row(0, named=True)
u, ini, fin = caso["user"], caso["start"], caso["end"]
print(f"Insider: {u} | ventana del ataque: {ini} → {fin}")

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

for ax, df, titulo in [
    (axes[0], logon.filter((pl.col("user") == u) & (pl.col("activity") == "Logon")), "Logins"),
    (axes[1], device.filter((pl.col("user") == u) & (pl.col("activity") == "Connect")), "Conexiones USB"),
]:
    ax.scatter(df["date"], df["date"].dt.hour(), s=12, alpha=0.6)
    ax.axvspan(ini, fin, alpha=0.15, color="red", label="Ventana del ataque")
    ax.axhspan(WORK_HOURS[0], WORK_HOURS[1], alpha=0.08, color="green")
    ax.set_ylabel("Hora del día"); ax.set_title(f"{titulo} — {u}")
    ax.set_ylim(-1, 24); ax.legend(loc="upper left")

plt.tight_layout(); plt.show()

## 8. Conclusiones del EDA

*(Completar tras ejecutar — plantilla de lo que hay que validar)*

- [ ] Nº de usuarios y rango temporal confirmados (≈1.000 usuarios, ene-2010 → may-2011).
- [ ] La distribución horaria justifica `WORK_HOURS = (8, 18)` — ¿ajustar?
- [ ] El insider del escenario 1 muestra el patrón esperado (USB+nocturno solo en la ventana roja).
- [ ] Las bajas de LDAP coinciden con el final de los escenarios 1 y 2.
- [ ] % de emails con destinatario externo — base para la feature de exfiltración.

**Siguiente fase (02_features):** construir la tabla usuario-día con las features
definidas en PLAN.md y guardarla en `PROCESSED_DIR` como parquet.